# Data Vortex – Aaruush'26 | Round 1
## Dataset 01 Cleaning & EDA

This notebook documents the reproducible recovery-data cleaning workflow. It does not fabricate source records; invalid values are either standardized, represented as `Unknown`/`[Missing text]`, or median-imputed for numeric engagement fields.

In [ ]:
import pandas as pd, numpy as np, html
from pathlib import Path

USERS_FILE = "Social_Engine_Users.csv"
POSTS_FILE = "Social_Engine_Posts_Corrupted.csv"

users = pd.read_csv(USERS_FILE)
posts = pd.read_csv(POSTS_FILE)

print("Users:", users.shape)
print("Posts:", posts.shape)

In [ ]:
# Users cleaning
users_clean = users.copy()
for c in ["location", "language"]:
    users_clean[c] = users_clean[c].astype("string").str.strip()
users_clean["language"] = users_clean["language"].str.lower()
users_clean["account_created"] = pd.to_datetime(users_clean["account_created"], errors="coerce")
users_clean = users_clean.drop_duplicates()
users_clean["account_created"] = users_clean["account_created"].dt.strftime("%Y-%m-%d")

print("Users after cleaning:", users_clean.shape)
print("Missing values:", users_clean.isna().sum())

In [ ]:
# Posts cleaning
p = posts.drop_duplicates().copy()

p["platform"] = p["platform"].astype("string").str.strip().fillna("Unknown")

p["text_content"] = p["text_content"].astype("string").map(
    lambda x: html.unescape(x) if pd.notna(x) else x
)
p["text_content"] = p["text_content"].str.replace(r"\s+", " ", regex=True).str.strip()
p.loc[p["text_content"].str.fullmatch(r"(?i)null", na=False), "text_content"] = pd.NA
p["text_content"] = p["text_content"].fillna("[Missing text]")

# Standardize mixed timestamp formats.
raw_ts = p["timestamp"].astype(str).str.strip()
dt = pd.Series(pd.NaT, index=p.index, dtype="datetime64[ns]")
num = pd.to_numeric(raw_ts, errors="coerce")
m = num.notna()
dt.loc[m] = pd.to_datetime(num[m], unit="s", errors="coerce")
m2 = ~m
dt.loc[m2] = pd.to_datetime(raw_ts[m2], format="%Y-%m-%dT%H:%M:%S", errors="coerce")
m3 = dt.isna()
dt.loc[m3] = pd.to_datetime(raw_ts[m3], format="%d-%m-%Y", errors="coerce")
p["timestamp"] = dt.dt.strftime("%Y-%m-%d %H:%M:%S")

# Engagement metrics: invalid negative values become missing, then median-imputed.
for c in ["likes", "shares", "comments"]:
    p[c] = pd.to_numeric(p[c], errors="coerce")
    p.loc[p[c] < 0, c] = np.nan
    p[c] = p[c].fillna(p[c].median()).round().astype(int)

print("Cleaned posts:", p.shape)
print(p.isna().sum())

In [ ]:
# Referential integrity
invalid_user_refs = (~p["user_id"].isin(users_clean["user_id"])).sum()
print("Invalid user references:", invalid_user_refs)

merged = p.merge(users_clean.assign(account_created=pd.to_datetime(users_clean["account_created"])),
                 on="user_id", how="left", validate="many_to_one")
merged["engagement_total"] = merged["likes"] + merged["shares"] + merged["comments"]
merged.head()

In [ ]:
# EDA
import matplotlib.pyplot as plt

print("Platform counts:")
print(merged["platform"].value_counts())

print("\nAverage engagement by platform:")
print(merged.groupby("platform")["engagement_total"].mean().sort_values(ascending=False))

print("\nTop locations by average engagement:")
print(merged.groupby("location")["engagement_total"].mean().sort_values(ascending=False).head(10))

print("\nFollower/likes correlation:")
print(merged[["follower_count","likes"]].corr().iloc[0,1])

merged["month"] = pd.to_datetime(merged["timestamp"]).dt.to_period("M").astype(str)
merged.groupby("month")["engagement_total"].sum().plot(kind="line", marker="o", figsize=(9,4))
plt.title("Monthly Total Engagement")
plt.xlabel("Month")
plt.ylabel("Total Engagement")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Save final cleaned datasets
users_clean.to_csv("Social_Engine_Users_Cleaned.csv", index=False)
p.to_csv("Social_Engine_Posts_Cleaned.csv", index=False)
print("Saved cleaned CSV files.")